In [1]:
import sys
import os
# Add build directory to path
build_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "python_lib")
sys.path.append(build_path)

from blackjack.blackjack_round import BJRound, BJStage, BJRules
from blackjack.actions import PlayerAction, DealerAction
from blackjack.cards import Card, Rank
import numpy as np
import time
import logging
from datetime import datetime
import os
from logging import FileHandler
import tqdm
from collections import deque
from blackjack.shoe import ProbabilisticRankShoe
import time
from blackjack.mixed_game_tree import MixedNode
import datetime
import matplotlib.pyplot as plt
from blackjack.game_tree_to_json import game_tree_to_json
import os
import pickle

In [2]:
rules = BJRules(
    dealer_checks_blackjack=True,
    dealer_hits_soft_17=False,
    allow_late_surrender=False,
    allow_early_surrender_on_ten=False,
    allow_early_surrender_on_ace=False,
    allow_early_surrender_on_all=False,
    dealer_shows_card_on_surrender=False,
    allow_insurance_vs_ace=True,
    natural_blackjack_payout=3/2,
    surrender_payout=1/2,
    insurance_payout=2/1,
    max_splits_allowed=1,
    allow_action_on_split_aces=True,
    allow_double_after_split=True,
    allow_double_on_soft=True,
    allow_split_different_tens=True,
    no_natural_bj_on_split=True
)

In [3]:
from collections import defaultdict
from collections import Counter
from blackjack.game_tree import SimulationResultNode


def iterate_nodes_by_levels(root_node):
    queue = deque()
    queue.append((root_node, 0))
    while queue:
        node, node_level = queue.popleft()
        yield node, node_level
        for child in node.children:
            queue.append((child, node_level + 1))


def get_nodes_by_levels(root_node, nodes_by_level_dict=None):
    if nodes_by_level_dict is None:
        nodes_by_level_dict = defaultdict(list)
    queue = deque()
    queue.append((root_node, 0))
    while queue:
        node, node_level = queue.popleft()
        nodes_by_level_dict[node_level].append(node)
        for child in node.children:
            queue.append((child, node_level + 1))
    return nodes_by_level_dict


def log_tree_structure(root_node):
    nodes_dict = get_nodes_by_levels(root_node)
    for lvl, nodes in nodes_dict.items():
        stage_count = count_stages(nodes)
        print(f"Level {lvl}: {len(nodes)} nodes", stage_count)


def count_stages(node_list):
    stage_counter = Counter()
    for node in node_list:
        if isinstance(node, SimulationResultNode):
            stage = BJStage.ROUND_OVER
        else:
            stage = node.bj_round.get_stage()
        
        if stage == BJStage.PLAYER_CARD:
            stage_name = "PLAYER_CARD"
            if len(node.children) == 1:
                stage_name += "_SINGLE"
            elif len(node.children) < 10:
                stage_name += "_PARTIAL"
            else:
                stage_name += "_FULL"
        else:
            stage_name = stage.name
        stage_counter[stage_name] += 1
    return stage_counter

In [4]:
main_actions_list = [
    PlayerAction.HIT,
    PlayerAction.STAND,
    PlayerAction.DOUBLE,
    PlayerAction.SPLIT
]


def best_action_hard_vs_dealer(hard_value, dealer_card):
    if hard_value < 5 or hard_value > 19:
        raise ValueError("Invalid hard value")

    if hard_value >= 12:
        player_card_0 = 10
    else:
        player_card_0 = 2
    
    player_card_1 = hard_value - player_card_0 

    cards = [
        Card(Rank.from_value(player_card_0)),
        Card(Rank.from_value(player_card_1)),
        Card(Rank.from_value(dealer_card))  
    ]
    
    bj_round = BJRound(rules)
    shoe = ProbabilisticRankShoe(8)
    bj_round.start_round(10)

    bj_round.take_card(cards[0])
    bj_round.take_card(cards[1])
    bj_round.take_card(cards[2])

    shoe.burn_rank_value(cards[0].rank_value())
    shoe.burn_rank_value(cards[1].rank_value())
    shoe.burn_rank_value(cards[2].rank_value())

    root_node = MixedNode(bj_round, shoe, monte_carlo_depth=6)
    root_node.build_tree()
    
    expected_value_0 = root_node.get_value()
    expected_value_1 = None

    main_action = None
    insurance_action = None
    action_node = root_node
    while main_action is None:
        stage = action_node.bj_round.get_stage()
        if stage == BJStage.DEALER_CHECK_BJ:
            # find branch where dealer doesn't have bj
            for ch in action_node.children:
                if ch.last_action == DealerAction.CONFIRM_NO_BLACKJACK:
                    action_node = ch
                    break
        elif stage == BJStage.PLAYER_OFFERED_INSURANCE:
            child_idx = np.argmax(action_node.children_prob)
            action_node = action_node.children[child_idx]
            insurance_action = action_node.last_action
        elif stage == BJStage.PLAYER_ACTION:
            action_child_idx = np.argmax(action_node.children_prob)
            after_action_node = action_node.children[action_child_idx]
            main_action = after_action_node.bj_round.last_action
            expected_value_1 = after_action_node.get_value()
        else:
            raise RuntimeError(f"Unexpected stage {stage}")
    
    return main_action, insurance_action, expected_value_0, expected_value_1

In [5]:
# action_data = {}

# for hard in tqdm.tqdm(range(5, 20)):
#     for dealer in range(2, 12):
#         result = best_action_hard_vs_dealer(hard, dealer)
#         action_data[(hard, dealer)] = result

# actions_only = {k: v[0] for k, v in action_data.items()}

# out_dir = "logs"
# out_path = os.path.join(out_dir, f"action_data.pkl")
# with open(out_path, "wb") as f:
#     pickle.dump(action_data, f, protocol=pickle.HIGHEST_PROTOCOL)

# actions_only_path = os.path.join(out_dir, f"actions_only.pkl")
# with open(actions_only_path, "wb") as f:
#     pickle.dump(actions_only, f, protocol=pickle.HIGHEST_PROTOCOL)


In [6]:
from blackjack.shoe import seed_shoe_rng

seed_shoe_rng(42)

bj_round = BJRound(rules)
shoe = ProbabilisticRankShoe(8)
bj_round.start_round(10)

cards = [
    Card(Rank.ACE),
    Card(Rank.ACE),
    Card(Rank.SIX) 
]

# cards = [
#     Card(Rank.TEN),
#     Card(Rank.SIX),
#     Card(Rank.ACE)  
# ]

cards = [c.rank_value() for c in cards]

bj_round.take_card(cards[0])
bj_round.take_card(cards[1])
bj_round.take_card(cards[2])

shoe.burn_rank_value(cards[0])
shoe.burn_rank_value(cards[1])
shoe.burn_rank_value(cards[2])

root_node = MixedNode(bj_round, shoe, max_hand_size_full_enum=1, player_card_initial_samples=1)

print(str(bj_round))

Last card: 6
Dealer 6X
Player AA(12/2)[$10]


In [7]:
for i in range(50):
    t0 = time.time()
    root_node.build_tree_layer(depth=i)
    t1 = time.time()
    seconds = np.round(t1 - t0)
    dt_whole = datetime.timedelta(seconds=seconds)
    print(f"Depth {i} built in {dt_whole} ({seconds} s)")
    if root_node.tree_completed():
        print("Tree completed")
        break

Depth 0 built in 0:00:00 (0.0 s)
Depth 1 built in 0:00:00 (0.0 s)
Depth 2 built in 0:00:00 (0.0 s)
Depth 3 built in 0:00:00 (0.0 s)
Depth 4 built in 0:00:00 (0.0 s)
Depth 5 built in 0:00:00 (0.0 s)
Depth 6 built in 0:00:01 (1.0 s)
Depth 7 built in 0:00:01 (1.0 s)
Depth 8 built in 0:00:01 (1.0 s)
Depth 9 built in 0:00:01 (1.0 s)
Depth 10 built in 0:00:01 (1.0 s)
Depth 11 built in 0:00:01 (1.0 s)
Depth 12 built in 0:00:01 (1.0 s)
Depth 13 built in 0:00:01 (1.0 s)
Depth 14 built in 0:00:01 (1.0 s)
Depth 15 built in 0:00:00 (0.0 s)
Depth 16 built in 0:00:00 (0.0 s)
Depth 17 built in 0:00:00 (0.0 s)
Depth 18 built in 0:00:00 (0.0 s)
Depth 19 built in 0:00:00 (0.0 s)
Depth 20 built in 0:00:00 (0.0 s)
Depth 21 built in 0:00:00 (0.0 s)
Depth 22 built in 0:00:00 (0.0 s)
Tree completed


In [8]:
print(root_node.get_value())

17.24457909212723


In [9]:
print(root_node.children_events)
print(root_node.children_prob)

[<PlayerAction.STAND: 'STAND'>, <PlayerAction.HIT: 'HIT'>, <PlayerAction.DOUBLE: 'DOUBLE'>, <PlayerAction.SPLIT: 'SPLIT'>]
[0, 0, 0, 1]


In [10]:
[ch.get_value() for ch in root_node.children]

[np.float64(-1.2),
 np.float64(6.4),
 np.float64(0.4),
 np.float64(17.24457909212723)]

In [11]:
root_node.children_events

[<PlayerAction.STAND: 'STAND'>,
 <PlayerAction.HIT: 'HIT'>,
 <PlayerAction.DOUBLE: 'DOUBLE'>,
 <PlayerAction.SPLIT: 'SPLIT'>]

In [12]:
# Last card: 6
# Dealer 6X
# Player AA(12/2)[$10]
# Depth 0 built in 0:00:00 (0.0 s)
# Depth 1 built in 0:00:00 (0.0 s)
# Depth 2 built in 0:00:00 (0.0 s)
# Depth 3 built in 0:00:00 (0.0 s)
# Depth 4 built in 0:00:00 (0.0 s)
# Depth 5 built in 0:00:00 (0.0 s)
# Depth 6 built in 0:00:01 (1.0 s)
# Depth 7 built in 0:00:02 (2.0 s)
# Depth 8 built in 0:00:02 (2.0 s)
# Depth 9 built in 0:00:02 (2.0 s)
# Depth 10 built in 0:00:02 (2.0 s)
# Depth 11 built in 0:00:02 (2.0 s)
# Depth 12 built in 0:00:02 (2.0 s)
# Depth 13 built in 0:00:01 (1.0 s)
# Depth 14 built in 0:00:01 (1.0 s)
# Depth 15 built in 0:00:01 (1.0 s)
# Depth 16 built in 0:00:00 (0.0 s)
# Depth 17 built in 0:00:00 (0.0 s)
# Depth 18 built in 0:00:00 (0.0 s)
# Depth 19 built in 0:00:00 (0.0 s)
# Depth 20 built in 0:00:00 (0.0 s)
# Depth 21 built in 0:00:00 (0.0 s)
# Tree completed

In [13]:
log_tree_structure(root_node)

Level 0: 1 nodes Counter({'PLAYER_ACTION': 1})
Level 1: 4 nodes Counter({'PLAYER_CARD_SINGLE': 2, 'DEALER_CARD': 1, 'PLAYER_CARD_FULL': 1})
Level 2: 13 nodes Counter({'PLAYER_CARD_FULL': 10, 'ROUND_OVER': 1, 'PLAYER_ACTION': 1, 'DEALER_CARD': 1})
Level 3: 103 nodes Counter({'PLAYER_ACTION': 99, 'DEALER_CARD': 2, 'PLAYER_CARD_SINGLE': 1, 'ROUND_OVER': 1})
Level 4: 300 nodes Counter({'PLAYER_CARD_SINGLE': 198, 'PLAYER_ACTION': 82, 'DEALER_CARD': 18, 'ROUND_OVER': 2})
Level 5: 461 nodes Counter({'PLAYER_ACTION': 180, 'PLAYER_CARD_SINGLE': 163, 'DEALER_CARD': 100, 'ROUND_OVER': 18})
Level 6: 711 nodes Counter({'PLAYER_CARD_SINGLE': 268, 'DEALER_CARD': 195, 'PLAYER_ACTION': 148, 'ROUND_OVER': 100})
Level 7: 833 nodes Counter({'DEALER_CARD': 254, 'PLAYER_CARD_SINGLE': 222, 'ROUND_OVER': 195, 'PLAYER_ACTION': 162})
Level 8: 829 nodes Counter({'ROUND_OVER': 254, 'DEALER_CARD': 235, 'PLAYER_CARD_SINGLE': 191, 'PLAYER_ACTION': 149})
Level 9: 769 nodes Counter({'ROUND_OVER': 235, 'DEALER_CARD': 2

In [14]:
for i in range(50):
    t0 = time.time()
    root_node.convert_to_full_up_to_depth(depth=i)
    t1 = time.time()
    seconds = np.round(t1 - t0)
    dt_whole = datetime.timedelta(seconds=seconds)
    print()
    print(f"Depth {i} converted to full enumeration in {dt_whole} ({seconds} s)")
    print("Value estimate = ", root_node.get_value())
    log_tree_structure(root_node)


Depth 0 converted to full enumeration in 0:00:00 (0.0 s)
Value estimate =  17.24457909212723
Level 0: 1 nodes Counter({'PLAYER_ACTION': 1})
Level 1: 4 nodes Counter({'PLAYER_CARD_SINGLE': 2, 'DEALER_CARD': 1, 'PLAYER_CARD_FULL': 1})
Level 2: 13 nodes Counter({'PLAYER_CARD_FULL': 10, 'ROUND_OVER': 1, 'PLAYER_ACTION': 1, 'DEALER_CARD': 1})
Level 3: 103 nodes Counter({'PLAYER_ACTION': 99, 'DEALER_CARD': 2, 'PLAYER_CARD_SINGLE': 1, 'ROUND_OVER': 1})
Level 4: 300 nodes Counter({'PLAYER_CARD_SINGLE': 198, 'PLAYER_ACTION': 82, 'DEALER_CARD': 18, 'ROUND_OVER': 2})
Level 5: 461 nodes Counter({'PLAYER_ACTION': 180, 'PLAYER_CARD_SINGLE': 163, 'DEALER_CARD': 100, 'ROUND_OVER': 18})
Level 6: 711 nodes Counter({'PLAYER_CARD_SINGLE': 268, 'DEALER_CARD': 195, 'PLAYER_ACTION': 148, 'ROUND_OVER': 100})
Level 7: 833 nodes Counter({'DEALER_CARD': 254, 'PLAYER_CARD_SINGLE': 222, 'ROUND_OVER': 195, 'PLAYER_ACTION': 162})
Level 8: 829 nodes Counter({'ROUND_OVER': 254, 'DEALER_CARD': 235, 'PLAYER_CARD_SINGLE

KeyboardInterrupt: 

In [ ]:
# root_node.convert_to_full_next_layer()
for i in range(98):
    root_node.single_node_from_sample_to_full()

In [ ]:
print(root_node.get_value()) # 16.86824325912692

In [ ]:
# with open("logs/game_tree.json", "w") as f:
#     game_tree_to_json(f, root_node)

In [ ]:
values = []

values.append(root_node.get_value())

for i in tqdm.tqdm(range(1000)):
    root_node.resample_player_cards()
    values.append(root_node.get_value())
mean_value = np.mean(values)

f, ax = plt.subplots()
ax.scatter(range(len(values)), values)
ax.hlines(mean_value, xmin=0, xmax=len(values), color="red", label=f"mean={mean_value:.3f}")
ax.legend()

In [ ]:
print(root_node.get_value())
print(root_node.children_prob)
print([f"{ch.get_value():.2f}" for ch in root_node.children])

In [ ]:
np.min(values), np.max(values)

In [ ]:
values = []

values.append(root_node.get_value())

for i in tqdm.tqdm(range(1000)):
    root_node.resample_player_cards()
    values.append(root_node.get_value())
mean_value = np.mean(values)


In [ ]:
f, ax = plt.subplots()
ax.scatter(range(len(values)), values)
ax.hlines(mean_value, xmin=0, xmax=len(values), color="red", label=f"mean={mean_value:.3f}")
ax.legend()

In [ ]:
np.min(values), np.max(values)

In [ ]:
for lvl in range(25):
    level_nodes = get_nodes_on_the_level(root_node, lvl)
    print(f"Level {lvl} has {len(level_nodes)} nodes")
    finished = False
    for n in level_nodes:
        if n.bj_round.get_stage() == BJStage.ROUND_OVER:
            continue
        elif isinstance(n, MonteCarloNode):
            print(f"MonteCarloNode found on level {lvl}")
            finished = True
            break
        else:
            break
    
    if finished:
        break

In [ ]:
from collections import Counter
count = Counter()

player_card_nodes = []
player_action_nodes = []
for node in level_nodes:
    stage = node.bj_round.get_stage()
    count[stage] += 1
    if stage == BJStage.PLAYER_CARD:
        player_card_nodes.append(node)
    if stage == BJStage.PLAYER_ACTION:
        player_action_nodes.append(node)



In [ ]:
count

In [ ]:
for i, n in enumerate(player_action_nodes):
    print(f"idx = {i}")
    print(f"Value = {n.get_value()}")
    print(str(n.bj_round))
    print()

In [ ]:
this_node = player_card_nodes[-1].parent
this_node.build_tree()

print(f"this_node Value = {this_node.get_value()}")
print(str(this_node.bj_round))
first_value = this_node.get_value()

In [ ]:
for i in range(100):
    changed = this_node.resample_player_cards()
    if this_node.get_value() == first_value:
        continue
    print(changed, this_node.get_value())
    print("-" * 32)

    # for i in range(len(this_node.children)):
    #     p = this_node.children_prob[i]
    #     ch = this_node.children[i]
    #     print(f"Value = {ch.get_value()}")
    #     print(f"Probability = {p}")
    #     print(str(ch.bj_round))
    #     print()
    # print("-" * 32)


In [ ]:
# this_node.has_completed_tree = False
# this_node.has_built_children = False
# this_node.children_prob = []
# this_node.children = []

In [ ]:
print(this_node.children[1].children[-1].bj_round)

In [ ]:
for i in range(100):
    this_node.resample_player_cards()
    children_of_interest = this_node.children[1].children[-1].children

    # if this_node.children_prob[1] == 0:
    #    continue
    
    # print(this_node.get_value())
    # print([f"{ch.get_value():.2f}" for ch in this_node.children])
    print(this_node.children_prob)

In [ ]:
this_node.resample_player_cards()

for ch in this_node.children:
    print(ch.get_value())
    print(str(ch.bj_round))
    print()

In [ ]:

print(this_node.children[0].get_value())

In [ ]:
print(this_node.children)